# Train the PI-RM for solving Diffusion equation

The 2D diffusion equation is:
$$\frac{\partial u}{\partial t}-\frac{\partial^{2}u}{\partial x^{2}}=f(t,x),$$
where $f(t,x)$ is the source term which was chosen equal to
$$f(t,x)=(\pi^{2}-1)\exp(-t)\sin(\pi x).$$
Additionally, $\Omega=[0,1]\times[0,1]$ is the chosen domain, with the following boundary conditions:
$$u(t = 0,x) = \sin(\pi x),$$
$$u(t,x = 0) = u(t,x = 1) = 0.$$
The exact solution of this PDE is
$$u(t,x) = \sin(\pi x)\exp(-t).$$

In [2]:
import torch
from rkan import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = RMultKAN(width=[2,5,1], grid=5, k=5, seed=2, device=device).speed()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

cuda
checkpoint directory created: ./model
saving model version 0.0


In [3]:
from torch import autograd
from tqdm import tqdm

# --- PDE and Solution Definitions ---
dim = 2
np_t = 100  # Number of sampling points in t dimension
np_x = 100  # # Number of sampling points in x dimension
np_b = 2000  # Number of sampling points per boundary
ranges_t = [0, 1]
ranges_x = [0, 1]

f_fun = lambda x: (torch.pi ** 2 - 1) * torch.exp(-x[:,[0]]) * torch.sin(torch.pi * x[:, [1]])
# exact solution
sol_fun = lambda x: torch.sin(torch.pi * x[:, [1]]) * torch.exp(-x[:,[0]])
init_fun = lambda x: torch.sin(torch.pi * x[:, [1]])
boundary_fun = lambda x: torch.zeros_like(x[:, [0]])

# Define the gradient calculation function
def batch_jacobian(func, x, create_graph=False):
    def _func_sum(x):
        return func(x).sum(dim=0)

    return autograd.functional.jacobian(_func_sum, x, create_graph=create_graph).permute(1, 0, 2)

# --- Data Sampling ---
# Internal sampling points
sampling_mode = 'random'
x_mesh = torch.linspace(ranges_t[0], ranges_t[1], steps=np_t)
y_mesh = torch.linspace(ranges_x[0], ranges_x[1], steps=np_x)
X, Y = torch.meshgrid(x_mesh, y_mesh, indexing="ij")
if sampling_mode == 'mesh':
    x_i = torch.stack([X.reshape(-1, ), Y.reshape(-1, )]).permute(1, 0)
else:
    x_i = torch.rand((np_t * np_x, 2))
x_i = x_i.to(device)

# Boundary points
x_b = torch.rand((np_b, 2))

t_bound_left = torch.stack([torch.full_like(x_b[:, 0], 0), x_b[:, 1]], dim=1).to(device)  # t=0

x_bound_lower = torch.stack([x_b[:, 0], torch.full_like(x_b[:, 1], 0)], dim=1).to(device)  # x=0
x_bound_upper = torch.stack([x_b[:, 0], torch.full_like(x_b[:, 1], 1)], dim=1).to(device)   # x=1

# --- Training Parameters ---
steps = 1000
alpha = 0.01
log = 1
loss_history = []
num_interval = model.grid
interval_size = np_t * np_x // num_interval

best_l2 = float('inf')
init_epoch = 0

def train():
    pbar = tqdm(range(steps), desc='Training', ncols=100)
    
    for _ in pbar:
        def closure():
            global pde_loss, bc_loss, lap, loss
            optimizer.zero_grad()
    
            # Interior loss
            x_1 = x_i.clone().detach().requires_grad_(True)
            sol = model(x_1)
            sol_t = batch_jacobian(model, x_1, create_graph=True)[:, :, 0]
            sol_xx = batch_jacobian(lambda x: batch_jacobian(model, x, create_graph=True)[:, :, 1], x_1, create_graph=True)[:, :, 1]
            
            f_val = f_fun(x_1)
            lap = sol_t - sol_xx - f_val
            pde_loss = torch.mean(lap ** 2)
    
            # Boundary loss
            init_pred = model(t_bound_left)
            init_exact = init_fun(t_bound_left)
            
            bc_lower_pred = model(x_bound_lower)
            bc_upper_pred = model(x_bound_upper)
            
            bc_loss = torch.mean((init_pred - init_exact)**2) + \
                      torch.mean(bc_lower_pred ** 2) + torch.mean(bc_upper_pred ** 2)
                
            loss = alpha * pde_loss + bc_loss
            loss.backward()
            return loss
       
        optimizer.step(closure)

        exact = sol_fun(x_i).to(device)
        pred_sol = model(x_i)
        l2 = torch.norm(exact - pred_sol) / torch.norm(exact)

        # auto save optimal model
        global best_l2
        if l2 < best_l2:
            best_l2 = l2
            model.saveckpt('./model/Dif_rkan')
        # Log progress
        if _ % log == 0:
            pbar.set_description("Lf: %.2e | Lb: %.2e | error: %.2e" % (
                pde_loss.cpu().detach().numpy(),
                bc_loss.cpu().detach().numpy(),
                l2.cpu().detach().numpy(),
            ))

        # Save loss history
        loss_history.append((pde_loss.cpu().detach().numpy(), bc_loss.cpu().detach().numpy(), loss.cpu().detach().numpy(), l2.cpu().detach().numpy()))

In [4]:
steps = 1000
for i in range(6):
    train()
    init_epoch = init_epoch + steps
    r_l2 = [loss[3] for loss in loss_history]
    print(f"Epoch {np.argmin(r_l2)}: Minimum Rel-L2 error reduced to {np.min(r_l2)*100: .4f}%")
    for param_group in optimizer.param_groups:
        param_group['lr'] *= 0.7
        print(f"Epoch {init_epoch}: Learning rate reduced to {param_group['lr']}")

Lf: 4.43e-04 | Lb: 1.37e-06 | error: 2.23e-03: 100%|████████████| 1000/1000 [02:39<00:00,  6.27it/s]


Epoch 999: Minimum Rel-L2 error reduced to  0.2228%
Epoch 1000: Learning rate reduced to 0.006999999999999999


Lf: 6.84e-04 | Lb: 4.23e-05 | error: 4.43e-03: 100%|████████████| 1000/1000 [01:48<00:00,  9.20it/s]


Epoch 1961: Minimum Rel-L2 error reduced to  0.1380%
Epoch 2000: Learning rate reduced to 0.004899999999999999


Lf: 8.58e-05 | Lb: 1.63e-07 | error: 8.32e-04: 100%|████████████| 1000/1000 [01:51<00:00,  8.98it/s]


...

Epoch 3999: Minimum Rel-L2 error reduced to  0.0529%
Epoch 4000: Learning rate reduced to 0.002400999999999999


Lf: 4.07e-05 | Lb: 5.04e-08 | error: 3.89e-04: 100%|████████████| 1000/1000 [01:45<00:00,  9.47it/s]


Epoch 4998: Minimum Rel-L2 error reduced to  0.0382%
Epoch 5000: Learning rate reduced to 0.0016806999999999992


Lf: 3.30e-05 | Lb: 4.18e-08 | error: 1.00e-03: 100%|████████████| 1000/1000 [01:43<00:00,  9.66it/s]

Epoch 5998: Minimum Rel-L2 error reduced to  0.0306%
Epoch 6000: Learning rate reduced to 0.0011764899999999994
